# Save tagged-unit figures

Independent notebook that loads one opto-tagging session, selects **robustly tagged units**,
and saves **one figure per unit**. Every condition for a unit is drawn in the **same figure**
(3-column grid), with the unit's mean waveform as an inset in the top-right corner. Figures are
saved in two alignments:

- **Pulse-aligned** (step 5) — aligned to each pulse; each condition panel is annotated with the
  pooled `reliability`, `median_latency`, `jitter` and `effect_ratio` for that unit/condition.
- **Train-aligned** (step 6) — aligned to each laser train, shading every pulse so the whole-train
  response is visible.

Tagging is split into two independent steps so metrics are computed once and selection
thresholds can be explored freely:

1. **Compute & save** (`OpticalTagging.compute_tagging_metrics`) — calculates the raw metrics
   for every unit × condition and, with `per_pulse=True`, **separately for each pulse position
   within the train** (`pulse_index = -1` is the pooled aggregate; `0, 1, 2, …` are individual
   pulses). Each row carries a `session` column (the NWB `session_id`) and the table is saved to
   a per-session CSV.
2. **Select** (`select_tagged_units`, a **standalone module-level function**) — applies the
   threshold criteria and adds the `tagged` flag. Re-run this alone to try new cut-offs. This
   notebook handles a **single session**; the saved CSVs can still be pooled across sessions and
   selected together elsewhere, since the function needs no `OpticalTagging` instance.

The layered criteria (on top of a paired t-test) avoid over-calling tagged units when trial
counts are large:

- **Short-latency window** (0–6 ms) for direct activation
- **SALT test** (Kvitsiani et al. 2013) on first-spike-latency distributions
- **Effect size**: `stim_FR ≥ k × baseline_FR`
- **Reliability**: fraction of pulses evoking a spike in the window
- **Latency / jitter**: small median first-spike latency and low SD
- **Multiple-comparison correction** (Benjamini–Hochberg FDR) on both the t-test and SALT
  p-values, applied **within each `pulse_index` family**

Figures are saved as **PNG by default** and rendered **in parallel** across worker processes.

In [1]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

✅ Analysis modules loaded from: /root/capsule/src/aind_dft_ephys_analysis


In [2]:
# =============================================================================
# 2. LOAD SESSION
# =============================================================================
import glob
from optical_tagging import OpticalTagging

# glutamatergic neurons
session_date = '863973_2026-08-25_14-18-29'
# GABAergic neurons
session_date = '868666_2026-09-25_14-50-29'

behavior_json_file = f'/root/capsule/data/ecephys_{session_date}/behavior/{session_date}.json'



# The sorted folder has an extra timestamp not contained in session_date,
# so locate it by glob (one match expected per session).
ephys_nwb_file = glob.glob(
    f'/root/capsule/data/ecephys_{session_date}_sorted_*/nwb/ecephys_{session_date}_experiment1_recording1.nwb'
)[0]

OptoTag = OpticalTagging(
    behavior_json_file=behavior_json_file,
    ephys_nwb_file=ephys_nwb_file,
)

# Where the computed (uncorrected) tagging metrics will be saved.
metrics_csv_path = f'/root/capsule/scratch/opto_tagging/metrics_{session_date}.csv'
print(f"OptoTag ready. Metrics CSV: {metrics_csv_path}")

Behavior data successfully loaded from: /root/capsule/data/ecephys_868666_2026-09-25_14-50-29/behavior/868666_2026-09-25_14-50-29.json
Number of units passing the default QC: 4029
OptoTag ready. Metrics CSV: /root/capsule/scratch/opto_tagging/metrics_868666_2026-09-25_14-50-29.csv


In [3]:
# =============================================================================
# 3. COMPUTE & SAVE TAGGING METRICS (calculation only, no selection)
#    Runs the SALT / latency / jitter / reliability / effect-size metrics for
#    every unit x condition and, with per_pulse=True, ALSO separately for each
#    pulse position within the train. Includes ALL probes / shanks / lasers.
#
#    Rows are labelled by `pulse_index`:
#       -1        -> pooled over ALL pulses in the train
#        0,1,2... -> that pulse position within the train only
#
#    These CALCULATION parameters define the metrics themselves; the SELECTION
#    thresholds are applied later in step 4 (so you can re-select without
#    recomputing).
# =============================================================================
import os
import pandas as pd

metrics_df = OptoTag.compute_tagging_metrics(
    baseline_window=(-0.05, 0.0),   # baseline rate / SALT baseline epochs
    latency_window=(0.0, 0.006),    # short direct-activation window (0-6 ms)
    salt_dt=0.001,                  # SALT latency-histogram resolution (s)
    per_pulse=True,                 # also compute per pulse position in the train
    align_to_event="pulse",
    remove_artefacts=True,
    removal_window=0.001,
)

# Persist the raw metrics so selection can be re-run without recomputation.
# The `session` column (= NWB session_id) lets these per-session CSVs be
# concatenated later and selected together in step 4.
os.makedirs(os.path.dirname(metrics_csv_path), exist_ok=True)
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Session: {OptoTag.session_name}")
print(f"Computed metrics for {len(metrics_df)} unit x condition x pulse rows.")
print(f"pulse_index values present: {sorted(metrics_df['pulse_index'].unique())}")
print(f"Saved metrics to: {metrics_csv_path}")

/opt/conda/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:430: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)


Session: ecephys_868666_2026-09-25_14-50-29
Computed metrics for 797742 unit x condition x pulse rows.
pulse_index values present: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Saved metrics to: /root/capsule/scratch/opto_tagging/metrics_868666_2026-09-25_14-50-29.csv


In [7]:
# =============================================================================
# 4. SELECT ROBUSTLY TAGGED UNITS (thresholding only; no recomputation)
#    `select_tagged_units` is a STANDALONE function (it does not need an
#    OpticalTagging instance). This notebook handles a SINGLE session, so we
#    select directly on the metrics_df from step 3. Change the thresholds below
#    and re-run this cell alone to explore different cut-offs.
# =============================================================================
from ast import literal_eval
from optical_tagging import select_tagged_units

# ---- Choose the condition to plot (all probes / all shanks are included) ----
# The condition is the 6-tuple:
#   (target_power, location_tag, laser_name, cycle_duration, frequency, pulse_duration)
#     5.0       -> target_power     : commanded laser power (mW)
#     1.0       -> location_tag      : stimulation location / site id
#     "Laser_1" -> laser_name        : which laser was used (IGNORED for selection)
#     1.0       -> cycle_duration    : laser-on duration of each train/cycle (s)
#     10.0      -> frequency         : pulse rate within the train (Hz) -> 10 pulses/s
#     0.05      -> pulse_duration    : duration of a single pulse (s) -> 50 ms
target_cond = (5.0, 1.0, "Laser_1", 1.0, 10.0, 0.05)

# ---- Choose which pulse position to select on -------------------------------
#   -1 -> pooled over ALL pulses in the train; 0 -> first pulse, 1 -> ...
SELECT_PULSE_INDEX = -1

# ---- Robust opto-tag SELECTION criteria (tune as needed) --------------------
#   effect_ratio   : require stim rate >= this x baseline rate
#   min_reliability: fraction of pulses that must evoke a spike in the window
#   max_latency    : max median first-spike latency (s)
#   max_jitter     : max SD of first-spike latency (s)
#   alpha          : applied to BH-FDR-corrected t-test AND SALT p-values
#                    (correction is done within each pulse_index family)
tagged_df = select_tagged_units(
    metrics_df,
    alpha=0.05,
    effect_ratio=2.0,
    min_abs_increase=0.0,
    min_reliability=0.2,
    max_latency=0.006,
    max_jitter=0.003,
    correction="fdr_bh",
)

# `condition` may be a string after a CSV round-trip; normalise for matching.
def as_tuple(c):
    if isinstance(c, str):
        try:
            c = literal_eval(c)
        except (ValueError, SyntaxError):
            return None
    return tuple(c) if isinstance(c, (tuple, list)) else None

# Match the condition on every field EXCEPT laser_name (index 2), so units tagged
# by any laser under the same power/location/timing are all included.
IGNORE_IDX = 2  # laser_name position in the 6-tuple
match_idx = [i for i in range(6) if i != IGNORE_IDX]

def matches_ignore_laser(c):
    c = as_tuple(c)
    if c is None or len(c) != 6:
        return False
    return all(c[i] == target_cond[i] for i in match_idx)

cond_mask = tagged_df["condition"].apply(matches_ignore_laser)
pulse_mask = tagged_df["pulse_index"] == SELECT_PULSE_INDEX

# Robustly tagged units for this condition / pulse (any probe / shank / laser).
sigrows = tagged_df[cond_mask & pulse_mask & (tagged_df["tagged"] == True)]

# Intersect with QC-passed units on the OptoTag object.
common_units = sorted(set(OptoTag.units_passing_qc.index) & set(sigrows["unit_id"]))
print(f"Condition (laser_name ignored): {target_cond} | pulse_index={SELECT_PULSE_INDEX}")
print(f"Robustly tagged units to plot ({len(common_units)}): {common_units}")

# Handy diagnostic table for the selected units.
cols = ["unit_id", "probe", "shank", "pulse_index", "reliability",
        "median_latency", "jitter", "baseline_FR_mean", "stim_FR_mean",
        "effect_ratio", "p_ttest_corr", "p_salt_corr"]
display(sigrows.sort_values("p_salt_corr")[cols].reset_index(drop=True))

Condition (laser_name ignored): (5.0, 1.0, 'Laser_2', 1.0, 10.0, 0.05) | pulse_index=-1
Robustly tagged units to plot (0): []


,unit_id,probe,shank,pulse_index,reliability,median_latency,jitter,baseline_FR_mean,stim_FR_mean,effect_ratio,p_ttest_corr,p_salt_corr


In [5]:
# =============================================================================
# 5. SAVE ONE FIGURE PER TAGGED UNIT — PULSE-ALIGNED (all conditions) — PARALLEL
# =============================================================================
# plot_raster_graph draws EVERY condition for a unit in a single figure
# (one column per condition), aligned to each PULSE. Passing metrics_df also
# annotates each condition panel with the pooled (pulse_index=-1) reliability,
# median_latency, jitter and effect_ratio for this unit. Rendering is the slow
# part (lazy NWB spike reads + matplotlib), so we fan units out over processes.
import os
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

from opto_tagging_figure_worker import init_worker, render_unit

# Output folder for the pulse-aligned per-unit figures.
pulse_fig_dir = f"/root/capsule/scratch/tagged_units/{session_date}/pulse_aligned/"
os.makedirs(pulse_fig_dir, exist_ok=True)

# By default we only save PNG. Add "eps"/"pdf"/"svg" here if you also need vectors.
SAVE_FORMATS = ["png"]

# Arguments forwarded to OptoTag.plot_raster_graph for every unit.
PULSE_PLOT_KWARGS = dict(
    bin_size=0.002,
    time_window=[-0.02, 0.09],
    removal_window=0.002,
    align_to_event="pulse",
    remove_artefacts=True,
    show_waveform=True,       # mean waveform in the top-right corner
    metrics_df=metrics_df,    # annotate reliability / latency / jitter / ratio
)

MAX_WORKERS = 4  # tune to the number of cores available in the capsule

ctx = mp.get_context("spawn")  # spawn avoids NWB/HDF5 fork-safety issues
init_args = (behavior_json_file, ephys_nwb_file, pulse_fig_dir, PULSE_PLOT_KWARGS, SAVE_FORMATS)

print(f"Rendering {len(common_units)} PULSE-aligned figures to {pulse_fig_dir}")
print(f"Using {MAX_WORKERS} worker processes | formats: {SAVE_FORMATS}")
with ProcessPoolExecutor(
    max_workers=MAX_WORKERS,
    mp_context=ctx,
    initializer=init_worker,
    initargs=init_args,
) as ex:
    for unit_id, ok, msg in ex.map(render_unit, common_units):
        print(f"  unit {unit_id}: {'saved' if ok else 'FAILED: ' + msg}")

print("Done saving pulse-aligned figures.")

Rendering 0 PULSE-aligned figures to /root/capsule/scratch/tagged_units/868666_2026-09-25_14-50-29/pulse_aligned/
Using 4 worker processes | formats: ['png']
Done saving pulse-aligned figures.


In [6]:
# =============================================================================
# 6. SAVE ONE FIGURE PER TAGGED UNIT — TRAIN-ALIGNED (all conditions) — PARALLEL
# =============================================================================
# Same units as step 5, but aligned to each LASER TRAIN (align_to_event="laser")
# so every pulse in the train is shaded and the full train response is visible.
# Per-pulse metrics are pulse-based, so they are NOT annotated here.
import os
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

from opto_tagging_figure_worker import init_worker, render_unit

# Output folder for the train-aligned per-unit figures.
train_fig_dir = f"/root/capsule/scratch/tagged_units/{session_date}/train_aligned/"
os.makedirs(train_fig_dir, exist_ok=True)

# Widen the window to span the whole train (cycle_duration = 1.0 s here).
TRAIN_PLOT_KWARGS = dict(
    bin_size=0.005,
    time_window=[-0.1, 1.2],
    removal_window=0.002,
    align_to_event="laser",   # align to each train instead of each pulse
    remove_artefacts=True,
    show_waveform=True,
)

ctx = mp.get_context("spawn")  # spawn avoids NWB/HDF5 fork-safety issues
init_args = (behavior_json_file, ephys_nwb_file, train_fig_dir, TRAIN_PLOT_KWARGS, SAVE_FORMATS)

print(f"Rendering {len(common_units)} TRAIN-aligned figures to {train_fig_dir}")
print(f"Using {MAX_WORKERS} worker processes | formats: {SAVE_FORMATS}")
with ProcessPoolExecutor(
    max_workers=MAX_WORKERS,
    mp_context=ctx,
    initializer=init_worker,
    initargs=init_args,
) as ex:
    for unit_id, ok, msg in ex.map(render_unit, common_units):
        print(f"  unit {unit_id}: {'saved' if ok else 'FAILED: ' + msg}")

print("Done saving train-aligned figures.")

Rendering 0 TRAIN-aligned figures to /root/capsule/scratch/tagged_units/868666_2026-09-25_14-50-29/train_aligned/
Using 4 worker processes | formats: ['png']
Done saving train-aligned figures.
